In [1]:
%load_ext autoreload
%autoreload 2

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
from hydra import compose, initialize

from main.model.script.hydra_beans import KdConfig

config_name = "train-local.yaml"
with initialize(version_base=None, config_path="../../../conf/"):
    cfg: KdConfig = compose(config_name="train-local.yaml")

/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'train-local.yaml': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)


To evaluate the good contribution of modalities we see MRR of it with and without the metric <br>
So if I have EEG, Aud, Txt and Vid and decide to ablate *vid* I measure:

A:MRR_mean modalities model that can use Vid but without video in inputstream <br>
B:MRR_mean across modalities of the ablated model

If B > A → Video is likely hurting other modalities<br>
If B < A → Video is likely helping them<br>
If B ~ A → Video has little effect on them<br>

In [3]:
from main.model.neegavi.factory import Factory
from main.model.neegavi.utils import get_model_ckpt

# TODO change?
baseline_checkpoint_path = "/home/jacopo/PycharmProjects/progetto-tesi/epochepoch=45-stepstep=117484.ckpt"
ckpt = get_model_ckpt(baseline_checkpoint_path)
baseline = Factory.best_inference().build()
baseline.load_state_dict(ckpt, strict=False)
baseline.eval()

EegInterAviModel(
  (pivot): ModalityStream(
    (adapter): EegAdapter(
      (ff): Sequential(
        (0): Linear(in_features=3800, out_features=384, bias=True)
        (1): GELU(approximate='none')
        (2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (supports): ModuleList(
    (0-1): 2 x ModalityStream(
      (adapter): PerceiverResamplerAdapter(
        (linear_reshape): Linear(in_features=768, out_features=384, bias=True)
        (resampler): PerceiverResampler(
          (blocks): ModuleList(
            (0-1): 2 x ModuleList(
              (0): PerceiverAttention(
                (norm_latents): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
                (norm_media): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
                (to_q): Linear(in_features=384, out_features=768, bias=False)
                (to_k): Linear(in_features=384, out_features=768, bias=False)
                (to_v): Linear(in_features=384, out_features=

In [4]:
import lightning
from main.model.neegavi.train_utils import KdTrainDataModule
from main.model.neegavi.training import EasyEegAviKdVateMaskedModule

trainer = lightning.Trainer(precision="16-mixed")

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


# Audio

In [5]:
from main.core_data.media.audio import Audio

audio_less_checkpoint_path = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-aud/2026-03-22_22-14-48/checkpoints/epochepoch=38-stepstep=99567.ckpt"

inference_ckpt = get_model_ckpt(audio_less_checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set([Audio.modality_code()])).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

EegInterAviModel(
  (pivot): ModalityStream(
    (adapter): EegAdapter(
      (ff): Sequential(
        (0): Linear(in_features=3800, out_features=384, bias=True)
        (1): GELU(approximate='none')
        (2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (supports): ModuleList(
    (0): ModalityStream(
      (adapter): PerceiverResamplerAdapter(
        (linear_reshape): Linear(in_features=768, out_features=384, bias=True)
        (resampler): PerceiverResampler(
          (blocks): ModuleList(
            (0-1): 2 x ModuleList(
              (0): PerceiverAttention(
                (norm_latents): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
                (norm_media): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
                (to_q): Linear(in_features=384, out_features=768, bias=False)
                (to_k): Linear(in_features=384, out_features=768, bias=False)
                (to_v): Linear(in_features=384, out_features=768, b

In [12]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

full_datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[baseline.pivot.code] + baseline.fusion_keys()
)

baseline_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=full_datamodule)
baseline_audioless_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=datamodule, )
audio_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )


baseline_results = trainer.validate(baseline_module, datamodule=full_datamodule)
baseline_audioless_results = trainer.validate(baseline_audioless_module, datamodule=datamodule)
audio_less_results = trainer.validate(audio_less_module, datamodule=datamodule)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.15890441834926605      │
│    val/fused/alignment_ecg    │      0.08251146972179413      │
│    val/fused/alignment_eeg    │      0.2187480628490448       │
│    val/fused/alignment_txt    │       0.136712446808815       │
│    val/fused/alignment_vid    │      0.16203391551971436      │
│     val/fused/margin_aud      │      0.2296687662601471       │
│     val/fused/margin_ecg      │      0.08898170292377472      │
│     val/fused/margin_eeg      │      0.2688747048377991       │
│     val/fused/margin_txt      │     0.007824438624083996      │
│     val/fused/margin_vid      │      0.21812927722930908      │
│ val/fused/meanR@1-3-5-10_aud  │      0.9527254104614258       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.5142778754234314       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9372175335884094       │
│ val/fused/meanR@1-3-5-10_mean │      0.6320019364356995       │
│ val/fused/meanR@1-3-5-10_txt  │     0.002206102479249239      │
│ val/fused/meanR@1-3-5-10_vid  │      0.7535828351974487       │
│       val/fused/mrr_aud       │       0.931821346282959       │
│       val/fused/mrr_ecg       │      0.4422060251235962       │
│       val/fused/mrr_eeg       │      0.9219871759414673       │
│      val/fused/mrr_mean       │      0.5905008912086487       │
│       val/fused/mrr_txt       │     0.0026500362437218428     │
│       val/fused/mrr_vid       │      0.6538397669792175       │
│      val/fused/top10_aud      │      0.9832521677017212       │
│      val/fused/top10_ecg      │      0.6598023176193237       │
│      val/fused/top10_eeg      │      0.9624455571174622       │
│     val/fused/top10_mean      │      0.7027074098587036       │
│      val/fused/top10_txt      │     0.004486987832933664      │
│      val/fused/top10_vid      │      0.9035499095916748       │
│      val/fused/top1_aud       │      0.9010353684425354       │
│      val/fused/top1_ecg       │      0.3278418481349945       │
│      val/fused/top1_eeg       │      0.8988413214683533       │
│      val/fused/top1_mean      │      0.5281620621681213       │
│      val/fused/top1_txt       │    0.00044869876001030207     │
│      val/fused/top1_vid       │      0.5126430988311768       │
│      val/fused/top3_aud       │      0.9546285271644592       │
│      val/fused/top3_ecg       │      0.49725425243377686      │
│      val/fused/top3_eeg       │      0.9377105236053467       │
│      val/fused/top3_mean      │      0.6300040483474731       │
│      val/fused/top3_txt       │     0.0013460962800309062     │
│      val/fused/top3_vid       │      0.7590807676315308       │
│      val/fused/top5_aud       │       0.971985399723053       │
│      val/fused/top5_ecg       │      0.5722130537033081       │
│      val/fused/top5_eeg       │       0.949872612953186       │
│      val/fused/top5_mean      │      0.6671342849731445       │
│      val/fused/top5_txt       │     0.0025426263455301523     │
│      val/fused/top5_vid       │      0.8390576839447021       │
│        val/fusion-loss        │       2.987936496734619       │
│        val/fusion/aud         │      2.7499730587005615       │
│        val/fusion/ecg         │       4.002493858337402       │
│        val/fusion/eeg         │       2.062603712081909       │
│        val/fusion/txt         │       4.173005104064941       │
│        val/fusion/vid         │      3.0062460899353027       │
│           val/loss            │       2.987936496734619       │
└───────────────────────────────┴───────────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_ecg    │       0.070527084171772       │
│    val/fused/alignment_eeg    │      0.1931627094745636       │
│    val/fused/alignment_txt    │      0.08452071994543076      │
│    val/fused/alignment_vid    │      0.17675046622753143      │
│     val/fused/margin_ecg      │      0.11738704144954681      │
│     val/fused/margin_eeg      │      0.24524736404418945      │
│     val/fused/margin_txt      │     0.013018516823649406      │
│     val/fused/margin_vid      │      0.2563571333885193       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.6742861270904541       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9158928394317627       │
│ val/fused/meanR@1-3-5-10_mean │      0.6027040481567383       │
│ val/fused/meanR@1-3-5-10_txt  │     0.0047861202619969845     │
│ val/fused/meanR@1-3-5-10_vid  │      0.8158512115478516       │
│       val/fused/mrr_ecg       │      0.6073072552680969       │
│       val/fused/mrr_eeg       │      0.8935176134109497       │
│      val/fused/mrr_mean       │      0.5580816864967346       │
│       val/fused/mrr_txt       │      0.00553309777751565      │
│       val/fused/mrr_vid       │      0.7259688377380371       │
│      val/fused/top10_ecg      │      0.7954421043395996       │
│      val/fused/top10_eeg      │      0.9515982866287231       │
│     val/fused/top10_mean      │      0.6735302209854126       │
│      val/fused/top10_txt      │     0.008525276556611061      │
│      val/fused/top10_vid      │      0.9385553002357483       │
│      val/fused/top1_ecg       │       0.503569483757019       │
│      val/fused/top1_eeg       │      0.8598076701164246       │
│      val/fused/top1_mean      │      0.49077343940734863      │
│      val/fused/top1_txt       │     0.001495662610977888      │
│      val/fused/top1_vid       │      0.5982208847999573       │
│      val/fused/top3_ecg       │      0.6683140993118286       │
│      val/fused/top3_eeg       │       0.91765958070755        │
│      val/fused/top3_mean      │       0.605137288570404       │
│      val/fused/top3_txt       │     0.0035895900800824165     │
│      val/fused/top3_vid       │      0.8309859037399292       │
│      val/fused/top5_ecg       │      0.7298187613487244       │
│      val/fused/top5_eeg       │      0.9345057010650635       │
│      val/fused/top5_mean      │      0.6413753032684326       │
│      val/fused/top5_txt       │     0.005533951334655285      │
│      val/fused/top5_vid       │      0.8956428170204163       │
│        val/fusion-loss        │      3.0600497722625732       │
│        val/fusion/ecg         │       4.012056827545166       │
│        val/fusion/eeg         │      2.3599910736083984       │
│        val/fusion/txt         │       4.172105312347412       │
│        val/fusion/vid         │       2.708749532699585       │
│           val/loss            │      3.0600497722625732       │
└───────────────────────────────┴───────────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_ecg    │      0.06030717492103577      │
│    val/fused/alignment_eeg    │      0.18398582935333252      │
│    val/fused/alignment_txt    │    -0.0012869583442807198     │
│    val/fused/alignment_vid    │      0.08579955250024796      │
│     val/fused/margin_ecg      │      0.14848056435585022      │
│     val/fused/margin_eeg      │      0.25881725549697876      │
│     val/fused/margin_txt      │      0.05861202999949455      │
│     val/fused/margin_vid      │      0.16615527868270874      │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8460323810577393       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9943298101425171       │
│ val/fused/meanR@1-3-5-10_mean │      0.6630415916442871       │
│ val/fused/meanR@1-3-5-10_txt  │      0.02508973889052868      │
│ val/fused/meanR@1-3-5-10_vid  │      0.7867144346237183       │
│       val/fused/mrr_ecg       │      0.7797389030456543       │
│       val/fused/mrr_eeg       │      0.9895931482315063       │
│      val/fused/mrr_mean       │      0.6219290494918823       │
│       val/fused/mrr_txt       │     0.025326823815703392      │
│       val/fused/mrr_vid       │      0.6930572390556335       │
│      val/fused/top10_ecg      │      0.9503020644187927       │
│      val/fused/top10_eeg      │      0.9995890855789185       │
│     val/fused/top10_mean      │      0.7309665679931641       │
│      val/fused/top10_txt      │      0.05115165933966637      │
│      val/fused/top10_vid      │      0.9228234887123108       │
│      val/fused/top1_ecg       │      0.6823174357414246       │
│      val/fused/top1_eeg       │       0.981756865978241       │
│      val/fused/top1_mean      │      0.5575058460235596       │
│      val/fused/top1_txt       │     0.005533951334655285      │
│      val/fused/top1_vid       │      0.5604150891304016       │
│      val/fused/top3_ecg       │       0.84788578748703        │
│      val/fused/top3_eeg       │       0.997288167476654       │
│      val/fused/top3_mean      │      0.6639470458030701       │
│      val/fused/top3_txt       │     0.015704456716775894      │
│      val/fused/top3_vid       │      0.7949097752571106       │
│      val/fused/top5_ecg       │       0.903624415397644       │
│      val/fused/top5_eeg       │      0.9986851811408997       │
│      val/fused/top5_mean      │      0.6997469663619995       │
│      val/fused/top5_txt       │     0.027968890964984894      │
│      val/fused/top5_vid       │      0.8687093257904053       │
│        val/fusion-loss        │      3.6153311729431152       │
│        val/fusion/ecg         │       4.079005241394043       │
│        val/fusion/eeg         │      2.4164841175079346       │
│        val/fusion/txt         │      4.9281392097473145       │
│        val/fusion/vid         │      3.7903544902801514       │
│           val/loss            │      3.6153311729431152       │
└───────────────────────────────┴───────────────────────────────┘

Main effect of audio: maybe positive, neutral, or slightly negative

# Txt

In [13]:
from main.core_data.media.text import Text

checkpoint_path = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-txt/2026-03-22_11-24-33/checkpoints/epochepoch=38-stepstep=99567.ckpt"

inference_ckpt = get_model_ckpt(checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set([Text.modality_code()])).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

EegInterAviModel(
  (pivot): ModalityStream(
    (adapter): EegAdapter(
      (ff): Sequential(
        (0): Linear(in_features=3800, out_features=384, bias=True)
        (1): GELU(approximate='none')
        (2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (supports): ModuleList(
    (0-1): 2 x ModalityStream(
      (adapter): PerceiverResamplerAdapter(
        (linear_reshape): Linear(in_features=768, out_features=384, bias=True)
        (resampler): PerceiverResampler(
          (blocks): ModuleList(
            (0-1): 2 x ModuleList(
              (0): PerceiverAttention(
                (norm_latents): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
                (norm_media): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
                (to_q): Linear(in_features=384, out_features=768, bias=False)
                (to_k): Linear(in_features=384, out_features=768, bias=False)
                (to_v): Linear(in_features=384, out_features=

In [14]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

baseline_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=full_datamodule)
baseline_modless_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=datamodule, )
audio_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )


baseline_results = trainer.validate(baseline_module, datamodule=full_datamodule)
baseline_modless_results = trainer.validate(baseline_modless_module, datamodule=datamodule)
audio_less_results = trainer.validate(audio_less_module, datamodule=datamodule)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.15890441834926605      │
│    val/fused/alignment_ecg    │      0.08251146972179413      │
│    val/fused/alignment_eeg    │      0.2187480628490448       │
│    val/fused/alignment_txt    │       0.136712446808815       │
│    val/fused/alignment_vid    │      0.16203391551971436      │
│     val/fused/margin_aud      │      0.2296687662601471       │
│     val/fused/margin_ecg      │      0.08898170292377472      │
│     val/fused/margin_eeg      │      0.2688747048377991       │
│     val/fused/margin_txt      │     0.007824438624083996      │
│     val/fused/margin_vid      │      0.21812927722930908      │
│ val/fused/meanR@1-3-5-10_aud  │      0.9527254104614258       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.5142778754234314       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9372175335884094       │
│ val/fused/meanR@1-3-5-10_mean │      0.6320019364356995       │
│ val/fused/meanR@1-3-5-10_txt  │     0.002206102479249239      │
│ val/fused/meanR@1-3-5-10_vid  │      0.7535828351974487       │
│       val/fused/mrr_aud       │       0.931821346282959       │
│       val/fused/mrr_ecg       │      0.4422060251235962       │
│       val/fused/mrr_eeg       │      0.9219871759414673       │
│      val/fused/mrr_mean       │      0.5905008912086487       │
│       val/fused/mrr_txt       │     0.0026500362437218428     │
│       val/fused/mrr_vid       │      0.6538397669792175       │
│      val/fused/top10_aud      │      0.9832521677017212       │
│      val/fused/top10_ecg      │      0.6598023176193237       │
│      val/fused/top10_eeg      │      0.9624455571174622       │
│     val/fused/top10_mean      │      0.7027074098587036       │
│      val/fused/top10_txt      │     0.004486987832933664      │
│      val/fused/top10_vid      │      0.9035499095916748       │
│      val/fused/top1_aud       │      0.9010353684425354       │
│      val/fused/top1_ecg       │      0.3278418481349945       │
│      val/fused/top1_eeg       │      0.8988413214683533       │
│      val/fused/top1_mean      │      0.5281620621681213       │
│      val/fused/top1_txt       │    0.00044869876001030207     │
│      val/fused/top1_vid       │      0.5126430988311768       │
│      val/fused/top3_aud       │      0.9546285271644592       │
│      val/fused/top3_ecg       │      0.49725425243377686      │
│      val/fused/top3_eeg       │      0.9377105236053467       │
│      val/fused/top3_mean      │      0.6300040483474731       │
│      val/fused/top3_txt       │     0.0013460962800309062     │
│      val/fused/top3_vid       │      0.7590807676315308       │
│      val/fused/top5_aud       │       0.971985399723053       │
│      val/fused/top5_ecg       │      0.5722130537033081       │
│      val/fused/top5_eeg       │       0.949872612953186       │
│      val/fused/top5_mean      │      0.6671342849731445       │
│      val/fused/top5_txt       │     0.0025426263455301523     │
│      val/fused/top5_vid       │      0.8390576839447021       │
│        val/fusion-loss        │       2.987936496734619       │
│        val/fusion/aud         │      2.7499730587005615       │
│        val/fusion/ecg         │       4.002493858337402       │
│        val/fusion/eeg         │       2.062603712081909       │
│        val/fusion/txt         │       4.173005104064941       │
│        val/fusion/vid         │      3.0062460899353027       │
│           val/loss            │       2.987936496734619       │
└───────────────────────────────┴───────────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.15771885216236115      │
│    val/fused/alignment_ecg    │      0.07847446948289871      │
│    val/fused/alignment_eeg    │      0.21497374773025513      │
│    val/fused/alignment_vid    │      0.16215617954730988      │
│     val/fused/margin_aud      │      0.22803254425525665      │
│     val/fused/margin_ecg      │      0.0886881947517395       │
│     val/fused/margin_eeg      │      0.26565635204315186      │
│     val/fused/margin_vid      │      0.2168823629617691       │
│ val/fused/meanR@1-3-5-10_aud  │      0.9524589776992798       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.5124931335449219       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9359643459320068       │
│ val/fused/meanR@1-3-5-10_mean │      0.7867664694786072       │
│ val/fused/meanR@1-3-5-10_vid  │      0.7461494207382202       │
│       val/fused/mrr_aud       │       0.930963933467865       │
│       val/fused/mrr_ecg       │      0.43950849771499634      │
│       val/fused/mrr_eeg       │      0.9208043813705444       │
│      val/fused/mrr_mean       │      0.7343525290489197       │
│       val/fused/mrr_vid       │      0.6461333632469177       │
│      val/fused/top10_aud      │      0.9834044575691223       │
│      val/fused/top10_ecg      │      0.6609005928039551       │
│      val/fused/top10_eeg      │      0.9613772630691528       │
│     val/fused/top10_mean      │      0.8759284019470215       │
│      val/fused/top10_vid      │      0.8980314135551453       │
│      val/fused/top1_aud       │      0.8995128273963928       │
│      val/fused/top1_ecg       │      0.3248215317726135       │
│      val/fused/top1_eeg       │      0.8976908326148987       │
│      val/fused/top1_mean      │      0.6566902995109558       │
│      val/fused/top1_vid       │      0.5047360062599182       │
│      val/fused/top3_aud       │      0.9549330472946167       │
│      val/fused/top3_ecg       │      0.4953322410583496       │
│      val/fused/top3_eeg       │      0.9363135695457458       │
│      val/fused/top3_mean      │      0.7842116355895996       │
│      val/fused/top3_vid       │      0.7502676844596863       │
│      val/fused/top5_aud       │       0.971985399723053       │
│      val/fused/top5_ecg       │      0.5689181685447693       │
│      val/fused/top5_eeg       │      0.9484755992889404       │
│      val/fused/top5_mean      │      0.8302354216575623       │
│      val/fused/top5_vid       │      0.8315624594688416       │
│        val/fusion-loss        │       2.744358539581299       │
│        val/fusion/aud         │      2.7735581398010254       │
│        val/fusion/ecg         │       4.042423248291016       │
│        val/fusion/eeg         │      2.1024625301361084       │
│        val/fusion/vid         │      3.0183165073394775       │
│           val/loss            │       2.744358539581299       │
└───────────────────────────────┴───────────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.07349353283643723      │
│    val/fused/alignment_ecg    │     0.055244795978069305      │
│    val/fused/alignment_eeg    │      0.18087665736675262      │
│    val/fused/alignment_vid    │      0.07645335048437119      │
│     val/fused/margin_aud      │      0.14455626904964447      │
│     val/fused/margin_ecg      │      0.10174447298049927      │
│     val/fused/margin_eeg      │      0.2159966081380844       │
│     val/fused/margin_vid      │      0.1449524462223053       │
│ val/fused/meanR@1-3-5-10_aud  │      0.9704629182815552       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8227622509002686       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9976168870925903       │
│ val/fused/meanR@1-3-5-10_mean │      0.8896530866622925       │
│ val/fused/meanR@1-3-5-10_vid  │      0.7677703499794006       │
│       val/fused/mrr_aud       │      0.9540208578109741       │
│       val/fused/mrr_ecg       │      0.7534230947494507       │
│       val/fused/mrr_eeg       │      0.9957916736602783       │
│      val/fused/mrr_mean       │       0.844876766204834       │
│       val/fused/mrr_vid       │      0.6762714982032776       │
│      val/fused/top10_aud      │      0.9917783737182617       │
│      val/fused/top10_ecg      │      0.9324547052383423       │
│      val/fused/top10_eeg      │      0.9996712803840637       │
│     val/fused/top10_mean      │      0.9581813812255859       │
│      val/fused/top10_vid      │      0.9088213443756104       │
│      val/fused/top1_aud       │       0.928745448589325       │
│      val/fused/top1_ecg       │      0.6518396735191345       │
│      val/fused/top1_eeg       │      0.9927684664726257       │
│      val/fused/top1_mean      │      0.7795915603637695       │
│      val/fused/top1_vid       │      0.5450127720832825       │
│      val/fused/top3_aud       │      0.9762485027313232       │
│      val/fused/top3_ecg       │       0.827018141746521       │
│      val/fused/top3_eeg       │      0.9987673163414001       │
│      val/fused/top3_mean      │      0.8935527205467224       │
│      val/fused/top3_vid       │      0.7721769213676453       │
│      val/fused/top5_aud       │      0.9850792288780212       │
│      val/fused/top5_ecg       │      0.8797364234924316       │
│      val/fused/top5_eeg       │      0.9992603659629822       │
│      val/fused/top5_mean      │      0.9272866249084473       │
│      val/fused/top5_vid       │      0.8450704216957092       │
│        val/fusion-loss        │      3.4331276416778564       │
│        val/fusion/aud         │      3.8342812061309814       │
│        val/fusion/ecg         │       4.200205326080322       │
│        val/fusion/eeg         │       2.517042875289917       │
│        val/fusion/vid         │       3.940300941467285       │
│           val/loss            │      3.4331276416778564       │
└───────────────────────────────┴───────────────────────────────┘

Text seems to fluctuate a lot from seed to seed. <br>
The modality is unstable.

> What if the increase in performance by removing audio is because it mitigates the presence of txt and thus removing modaltiies while txt is in it always proves gains?

# ECG

In [ ]:
from main.core_data.media.text import Text

audio_less_checkpoint_path = ""

inference_ckpt = get_model_ckpt(audio_less_checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set(Text.modality_code())).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

In [ ]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

full_datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[baseline.pivot.code] + baseline.fusion_keys()
)

baseline_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=full_datamodule)
baseline_modless_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=datamodule, )
audio_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )


baseline_results = trainer.validate(baseline_module, datamodule=full_datamodule)
baseline_modless_results = trainer.validate(baseline_modless_module, datamodule=datamodule)
audio_less_results = trainer.validate(audio_less_module, datamodule=datamodule)

In [ ]:
baseline_results

In [ ]:
audio_less_results